# 7 · Saddle-point problems 🐎

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=07-saddle-point.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/07-saddle-point.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>
:::


Welcome **into the saddle**. Part II opens with its mathematical namesake: the
**saddle-point problem**. It appears whenever a minimisation is subject to a
**constraint** — the constraint is enforced by a **Lagrange multiplier**, and the
pair (solution, multiplier) is a *saddle point* of the Lagrangian, not a minimum.

The classic example is **Stokes flow** — slow, viscous, incompressible fluid:
$$ -\Delta \mathbf u + \nabla p = \mathbf f,\qquad \operatorname{div}\mathbf u = 0 . $$
The velocity $\mathbf u$ wants to minimise the viscous energy, but is **constrained**
to be divergence-free (incompressible); the **pressure** $p$ is exactly the
multiplier enforcing that constraint.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
from netgen.occ import unit_square
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.05))

## 1. Two spaces, and a block system with a hole in it

We discretise velocity and pressure with the **Taylor–Hood** pair — vector $H^1$
of order $k$ for $\mathbf u$, scalar $H^1$ of order $k-1$ for $p$ (this pairing is
*inf-sup stable*: it does not lock). The weak form has three pieces,
$$ \underbrace{\int \nabla\mathbf u\!:\!\nabla\mathbf v}_{A}
   \;+\;\underbrace{\int \operatorname{div}\mathbf v\,p}_{B^\top}
   \;+\;\underbrace{\int \operatorname{div}\mathbf u\,q}_{B} \;=\; \int\mathbf f\!\cdot\!\mathbf v, $$
which assemble into a **block matrix** with a tell-tale **zero block**:
$$ K=\begin{pmatrix}A & B^\top\\ B & 0\end{pmatrix}. $$
That zero (no pressure–pressure coupling) is the signature of a saddle point.

In [ ]:
V = VectorH1(mesh, order=2, dirichlet="bottom|right|top|left")   # velocity
Q = H1(mesh, order=1)                                            # pressure
u, v = V.TnT()
p, q = Q.TnT()

A  = BilinearForm(InnerProduct(Grad(u), Grad(v)) * dx).Assemble()
B  = BilinearForm(trialspace=V, testspace=Q)
B += div(u) * q * dx
B.Assemble()
K = BlockMatrix([[A.mat, B.mat.T],
                 [B.mat, None]])           # the saddle-point operator
print("velocity dofs:", V.ndof, " pressure dofs:", Q.ndof)

## 2. Why we cannot just Cholesky it

$K$ is **symmetric** but **indefinite**: it has both positive and negative
eigenvalues (the pressure block sits on the diagonal as a *zero*, dragging
eigenvalues below zero). A Cholesky factorisation — our trusty
`inverse="sparsecholesky"` — **requires a positive-definite matrix** and would
break down on $K$. We need a solver built for **symmetric indefinite** systems.

## 3. MinRes with a block preconditioner

The **minimal-residual** method `MinRes` is the right Krylov solver for symmetric
indefinite systems (its cousin `CG` would silently misbehave — it assumes
positive definiteness). To make it converge fast we feed it a **block-diagonal
preconditioner** built from the two pieces that *are* positive definite:
$$ C=\begin{pmatrix} A^{-1} & 0\\ 0 & M_p^{-1}\end{pmatrix}, $$
the inverse of the velocity stiffness $A$ and the inverse of the **pressure mass
matrix** $M_p$. Both are SPD, so **each is a cheap `sparsecholesky`** — no
indefinite direct solver anywhere.

In [ ]:
mp = BilinearForm(p * q * dx).Assemble()                # pressure mass matrix
C = BlockMatrix([[A.mat.Inverse(V.FreeDofs(), inverse="sparsecholesky"), None],
                 [None, mp.mat.Inverse(inverse="sparsecholesky")]])

## 4. Drive the flow: a lid-driven cavity

The square box is closed; we drag the **lid** (the top edge) tangentially. To
keep the corners clean we use a profile that fades to zero where the lid meets the
fixed walls. We set the boundary velocity, move it to the right-hand side, and let
`MinRes` solve for the rest.

In [ ]:
gfu = GridFunction(V)
gfp = GridFunction(Q)
lid = CF((16 * x * x * (1 - x) * (1 - x), 0))           # 1 in the middle, 0 at the corners
gfu.Set(lid, definedon=mesh.Boundaries("top"))

rhs = BlockVector([(-A.mat * gfu.vec - B.mat.T * gfp.vec).Evaluate(),
                   (-B.mat * gfu.vec).Evaluate()])
du = gfu.vec.CreateVector(); du[:] = 0
dp = gfp.vec.CreateVector(); dp[:] = 0
with TaskManager():
    solvers.MinRes(mat=K, pre=C, rhs=rhs, sol=BlockVector([du, dp]),
                   maxsteps=500, tol=1e-10, printrates=False)
gfu.vec.data += du
gfp.vec.data += dp
gfp.Set(gfp - Integrate(gfp, mesh))                     # pin the pressure constant (zero mean)

print(f"divergence ‖div u‖ = {sqrt(Integrate(div(gfu)**2, mesh)):.1e}  (≈ incompressible)")

## 5. The flow and its pressure

The lid drags the fluid into a single big **recirculating vortex**; the pressure
multiplier adjusts everywhere to keep the flow divergence-free.

In [ ]:
Draw(gfu, mesh, "velocity", vectors={"grid_size": 26})

In [ ]:
Draw(Norm(gfu), mesh, "speed |u|")

In [ ]:
Draw(gfp, mesh, "pressure p (the constraint's multiplier)")

:::{dropdown} 📚 Further reading
:class: further-reading

- i-tutorial [2.6 Stokes](https://docu.ngsolve.org/latest/i-tutorials/unit-2.6-stokes/stokes.html).
- i-tutorial [3.2 incompressible Navier–Stokes](https://docu.ngsolve.org/latest/i-tutorials/unit-3.2-navierstokes/navierstokes.html).
:::

:::{dropdown} 🧠 Quiz — why `MinRes` and not `CG`, and why the *mass matrix* for the pressure block?
:class: quiz
`CG` is built for **symmetric positive-definite** systems; on our indefinite $K$
its underlying energy norm is not a norm and the iteration can break down or stall.
`MinRes` instead minimises the **residual** in the (genuine) 2-norm and is happy
with any **symmetric** matrix, definite or not. The preconditioner needs to be SPD,
so we cannot use $K$ itself — we approximate the **Schur complement**
$S=B A^{-1}B^\top$ (which governs the pressure) by the **pressure mass matrix**
$M_p$: for Stokes one can prove $M_p$ is *spectrally equivalent* to $S$, so
$M_p^{-1}$ is a cheap, mesh-independent pressure preconditioner. The result: a
constant number of `MinRes` steps, each using only SPD `sparsecholesky` solves.
:::

This block-preconditioning idea — split a hard system into pieces you *can* invert
— builds directly on the **solver toolbox** (notebook 6), now applied to a
saddle-point system. Next, we stay in the saddle and learn to label **materials and
boundaries**.

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "08-materials-boundaries", "8 · Materials, boundaries & labels"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))